[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/A.%20Foundations%20of%20Terminal%20Operations%20%28The%20Building%20Blocks%29/02.%20The%20Container%20Stacking%20Rules%20Problem/P2-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 2. The Container Stacking Rules Problem

## Tier 3 — Genetic Algorithm (GA) for near-optimal stacking

A constructive heuristic (Tier 2) is fast and interpretable, but it can be short-sighted.

Tier 3 introduces a **metaheuristic**: a Genetic Algorithm (GA). Instead of making one greedy decision at a time, the GA searches over many candidate stacking plans and tries to improve them over generations.

### Learning goals

- Understand how to encode a stacking plan as a **chromosome**.
- Define a clear **fitness function** (lower reshuffles is better).
- See how selection / crossover / mutation improves solutions.
- Interpret convergence curves.

### What this notebook outputs

- A best-found stack assignment (near-optimal for the small instance)
- Reshuffle (blocking) analysis
- GA convergence plot
- Logistics-focused visuals: stack diagram + blocking arrows + reshuffle breakdown

In [1]:
# Environment check (no installs here)
#
# Best practice: dependencies should be preinstalled in the JupyterHub/Docker image.
# If you're running locally, install packages once in your Python environment.

from typing import List, Tuple, Dict, Optional

try:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
except ImportError as e:
    raise ImportError(
        "Missing dependency. Install: numpy, pandas, matplotlib. "
        "If you use the provided JupyterHub Docker image, these should already be installed."
    ) from e

# Reproducible randomness for the GA
rng = np.random.default_rng(42)

print("Dependencies imported successfully.")

Dependencies imported successfully.


## Concrete example instance

We reuse the same small instance used earlier (5 containers, 3 stacks, 2 tiers). This keeps the GA code easy to understand.

- Containers: C1..C5
- Departure times: C1(d=3), C2(d=1), C3(d=5), C4(d=2), C5(d=6)
- (Illustrative) weights: C1(w=22), C2(w=20), C3(w=18), C4(w=25), C5(w=16)

### Objective

Minimize the number of blocking relationships ("reshuffles") implied by the final stack configuration.

In [2]:
# ----------------------------
# Example data
# ----------------------------
# We keep the instance small so the GA is easy to follow.

containers = [
    {"id": "C1", "d": 3, "w": 22},
    {"id": "C2", "d": 1, "w": 20},
    {"id": "C3", "d": 5, "w": 18},
    {"id": "C4", "d": 2, "w": 25},
    {"id": "C5", "d": 6, "w": 16},
]

NUM_STACKS = 3
MAX_TIERS = 2

# Convenience lookups
cid_to_d = {c["id"]: c["d"] for c in containers}
cid_to_w = {c["id"]: c["w"] for c in containers}

# Arrival order (as in the narrative): we process containers in this order
arrival_order = [c["id"] for c in containers]

arrival_order

In [ ]:
# ----------------------------
# Reshuffle / blocking metric (same simple definition as earlier Tiers)
# ----------------------------
# In one stack, containers are stored bottom -> top.
# A blocking pair exists when a lower container departs earlier than an upper container.
# Each blocking pair contributes 1 expected reshuffle.


def count_blocking_pairs_in_stack(stack_bottom_to_top: List[str]) -> int:
    reshuffles = 0
    for lower_idx in range(len(stack_bottom_to_top)):
        for upper_idx in range(lower_idx + 1, len(stack_bottom_to_top)):
            lower = stack_bottom_to_top[lower_idx]
            upper = stack_bottom_to_top[upper_idx]
            if cid_to_d[lower] < cid_to_d[upper]:
                reshuffles += 1
    return reshuffles


def count_total_reshuffles(stack_config: List[List[str]]) -> int:
    return sum(count_blocking_pairs_in_stack(stack) for stack in stack_config)


# Tiny sanity check
count_total_reshuffles([["C2", "C1"], [], []])  # C1 blocks C2 => 1

## Chromosome encoding (how GA represents a stacking plan)

We need a representation that is:

- easy to mutate / crossover
- easy to decode into a real stack configuration

### Encoding

A chromosome is a list of length **5** (one gene per container in arrival order).

- gene value ∈ {0,1,2}
- meaning: the *preferred stack index* for that container

Example:

- chromosome `[0, 1, 2, 1, 2]`
  - C1 prefers Stack 1
  - C2 prefers Stack 2
  - C3 prefers Stack 3
  - C4 prefers Stack 2
  - C5 prefers Stack 3

### Decoding rule

We process containers in arrival order. For each container:

- try to place it into its preferred stack
- if that stack is full, place it into the first stack with available space

This guarantees a feasible configuration.

In [ ]:
# ----------------------------
# Decode chromosome -> stack configuration
# ----------------------------
# This is the bridge from GA representation to a real stacking plan.


def decode_chromosome(chromosome: List[int]) -> List[List[str]]:
    """Decode a chromosome into a feasible stack configuration.

    - `chromosome[i]` is the preferred stack index (0..NUM_STACKS-1)
      for `arrival_order[i]`.

    Decoding rule:
    - place into preferred stack if it has capacity
    - otherwise place into the first stack with capacity
    """

    stacks: List[List[str]] = [[] for _ in range(NUM_STACKS)]

    for gene, cid in zip(chromosome, arrival_order):
        preferred = int(gene)

        # If preferred stack has space, use it.
        if len(stacks[preferred]) < MAX_TIERS:
            stacks[preferred].append(cid)
            continue

        # Otherwise, fallback to first available stack.
        placed = False
        for s_idx in range(NUM_STACKS):
            if len(stacks[s_idx]) < MAX_TIERS:
                stacks[s_idx].append(cid)
                placed = True
                break

        # With our dimensions (3 stacks x 2 tiers = 6 slots),
        # and 5 containers, we should always be able to place.
        assert placed

    return stacks


# Quick decode sanity check
decode_chromosome([0, 1, 2, 1, 2])

## Fitness function (what does the GA optimize?)

To evolve better stacking plans, the GA needs a numeric score.

We minimize a simple and transparent objective:

- **reshuffles = number of blocking pairs** in the decoded stack configuration

Optionally, we can add a small penalty for weight instability:

- if a heavier container sits above a lighter one in the same stack, add a penalty

This mimics the idea that real terminal rules often balance multiple operational criteria.

In [ ]:
# ----------------------------
# Fitness function
# ----------------------------
# Lower is better.
#
# Main term:
# - number of blocking pairs (expected reshuffles)
#
# Optional secondary term:
# - small penalty for weight instability (heavier above lighter)


def weight_instability_penalty(stack_config: List[List[str]]) -> float:
    """Return a small penalty for heavy-over-light patterns."""

    pen = 0.0

    for stack in stack_config:
        # Check adjacent pairs (bottom->top)
        for i in range(len(stack) - 1):
            lower = stack[i]
            upper = stack[i + 1]

            # If the upper container is heavier than the lower one,
            # add a penalty.
            if cid_to_w[upper] > cid_to_w[lower]:
                pen += 1.0

    return pen


def fitness(chromosome: List[int], weight_penalty_coef: float = 0.25) -> float:
    """Compute the objective value for a chromosome."""

    config = decode_chromosome(chromosome)

    resh = float(count_total_reshuffles(config))
    wpen = float(weight_instability_penalty(config))

    # Total objective (minimize)
    return resh + weight_penalty_coef * wpen


# Sanity check: fitness of a random chromosome
fitness([0, 1, 2, 1, 2]), fitness(rng.integers(0, NUM_STACKS, size=len(arrival_order)).tolist())

## Genetic Algorithm implementation

We use a small, readable GA with standard parts:

- **Initialize** a random population of chromosomes
- **Selection**: tournament selection (pick the best from a small random subset)
- **Crossover**: single-point crossover
- **Mutation**: randomly change a gene to another stack index
- **Elitism**: keep the best few solutions each generation

We log the best objective value over generations so we can plot convergence.

In [ ]:
# ----------------------------
# GA operators and main loop
# ----------------------------


def random_chromosome() -> List[int]:
    """Create a random chromosome."""
    return rng.integers(0, NUM_STACKS, size=len(arrival_order)).tolist()


def tournament_select(pop: List[List[int]], k: int = 3) -> List[int]:
    """Tournament selection (minimization).

    Pick k random individuals and return the one with the smallest fitness.
    """

    idxs = rng.choice(len(pop), size=k, replace=False)

    best = None
    best_val = float("inf")

    for i in idxs:
        v = fitness(pop[i])
        if v < best_val:
            best_val = v
            best = pop[i]

    assert best is not None
    return best


def crossover(p1: List[int], p2: List[int]) -> Tuple[List[int], List[int]]:
    """Single-point crossover."""

    n = len(p1)
    point = int(rng.integers(1, n))

    c1 = p1[:point] + p2[point:]
    c2 = p2[:point] + p1[point:]

    return c1, c2


def mutate(chrom: List[int], rate: float = 0.10) -> List[int]:
    """Mutate each gene with probability `rate` by replacing it with a random stack index."""

    out = chrom.copy()
    for i in range(len(out)):
        if rng.random() < rate:
            out[i] = int(rng.integers(0, NUM_STACKS))
    return out


def run_ga(
    pop_size: int = 60,
    generations: int = 200,
    elite: int = 4,
    crossover_rate: float = 0.8,
    mutation_rate: float = 0.10,
):
    # Initialize population
    pop = [random_chromosome() for _ in range(pop_size)]

    best_val = float("inf")
    best_chrom = None
    history_best = []

    for gen in range(generations):
        # Score population
        scored = [(fitness(ch), ch) for ch in pop]
        scored.sort(key=lambda x: x[0])

        # Track best
        if scored[0][0] < best_val:
            best_val = float(scored[0][0])
            best_chrom = scored[0][1]

        history_best.append(best_val)

        # Elitism: keep top `elite`
        new_pop = [ch for (_, ch) in scored[:elite]]

        # Fill the rest
        while len(new_pop) < pop_size:
            p1 = tournament_select(pop)
            p2 = tournament_select(pop)

            if rng.random() < crossover_rate:
                c1, c2 = crossover(p1, p2)
            else:
                c1, c2 = p1.copy(), p2.copy()

            c1 = mutate(c1, rate=mutation_rate)
            c2 = mutate(c2, rate=mutation_rate)

            new_pop.extend([c1, c2])

        pop = new_pop[:pop_size]

    assert best_chrom is not None

    return best_chrom, best_val, history_best


best_chrom, best_obj, history_best = run_ga()

print("Best chromosome:", best_chrom)
print("Best objective:", best_obj)
print("Decoded stack config:", decode_chromosome(best_chrom))

Best chromosome: [2, 2, 1, 1, 2]
Best objective: 0.25
Decoded stack config: [['C5'], ['C3', 'C4'], ['C1', 'C2']]


## GA convergence

We plot the best objective value found so far versus generation.

- If the curve goes down quickly and then flattens, the GA is converging.
- If it stays flat early, try increasing population size or mutation rate.

In [ ]:
# Plot best-so-far objective (lower is better)
plt.figure(figsize=(7, 3))
plt.plot(history_best, lw=2, color="#344054")
plt.title("GA convergence (best objective so far)")
plt.xlabel("Generation")
plt.ylabel("Best objective")
plt.grid(True, alpha=0.3)
plt.show()

## Best solution analysis and visualization

Now we take the best chromosome found by the GA and:

- decode it into a stack configuration
- compute blocking relationships (reshuffles)
- visualize the final stacks with blocking arrows
- plot reshuffles per container (retrieval order) + cumulative reshuffles

This makes the GA result operationally interpretable.

In [ ]:
# ----------------------------
# Decode best solution and compute blocking pairs
# ----------------------------

best_config = decode_chromosome(best_chrom)


def blocking_pairs(stack_config: List[List[str]]) -> pd.DataFrame:
    pairs = []
    for s_idx, stack in enumerate(stack_config, start=1):
        for lower_idx in range(len(stack)):
            for upper_idx in range(lower_idx + 1, len(stack)):
                lower = stack[lower_idx]
                upper = stack[upper_idx]
                if cid_to_d[lower] < cid_to_d[upper]:
                    pairs.append(
                        {
                            "stack": s_idx,
                            "lower": lower,
                            "upper": upper,
                            "lower_departure": cid_to_d[lower],
                            "upper_departure": cid_to_d[upper],
                        }
                    )
    return pd.DataFrame(pairs)


blocking_df = blocking_pairs(best_config)

print("Best decoded config:", best_config)
print("Blocking pairs (reshuffles):", int(len(blocking_df)))

blocking_df

Best decoded config: [['C5'], ['C3', 'C4'], ['C1', 'C2']]
Blocking pairs (reshuffles): 0


In [ ]:
# ----------------------------
# Visualize best solution: stacks + blocking arrows
# ----------------------------

fig, ax = plt.subplots(figsize=(8, 4))

stack_width = 1.0
cell_height = 1.0
x_gap = 0.6

stack_x = {s_idx: (s_idx - 1) * (stack_width + x_gap) for s_idx in range(1, NUM_STACKS + 1)}

# Draw empty slots
for s_idx in range(1, NUM_STACKS + 1):
    x0 = stack_x[s_idx]
    for tier in range(MAX_TIERS):
        y0 = tier * cell_height
        rect = plt.Rectangle((x0, y0), stack_width, cell_height, facecolor="#F2F4F7", edgecolor="#344054")
        ax.add_patch(rect)

# Labels
for s_idx, stack in enumerate(best_config, start=1):
    x0 = stack_x[s_idx]
    for tier, cid in enumerate(stack):
        y0 = tier * cell_height
        ax.text(
            x0 + stack_width / 2,
            y0 + cell_height / 2,
            f"{cid}\n(d={cid_to_d[cid]}, w={cid_to_w[cid]})",
            ha="center",
            va="center",
            fontsize=9,
            color="#101828",
        )

# Blocking arrows
if len(blocking_df) > 0:
    for _, row in blocking_df.iterrows():
        s_idx = int(row["stack"])
        lower = row["lower"]
        upper = row["upper"]

        stack = best_config[s_idx - 1]
        lower_tier = stack.index(lower)
        upper_tier = stack.index(upper)

        x = stack_x[s_idx] + stack_width / 2
        y_lower = lower_tier * cell_height + cell_height / 2
        y_upper = upper_tier * cell_height + cell_height / 2

        ax.annotate(
            "",
            xy=(x, y_lower),
            xytext=(x, y_upper),
            arrowprops=dict(arrowstyle="->", lw=2, color="#F97066"),
        )

ax.set_title("Tier 3 GA — best stacking plan (with blocking arrows)")
ax.set_xticks([stack_x[s] + stack_width / 2 for s in range(1, NUM_STACKS + 1)])
ax.set_xticklabels([f"Stack {s}" for s in range(1, NUM_STACKS + 1)])
ax.set_yticks([])
ax.set_xlim(-0.5, stack_x[NUM_STACKS] + stack_width + 0.5)
ax.set_ylim(-0.2, MAX_TIERS * cell_height + 0.8)
plt.show()

In [ ]:
# ----------------------------
# Reshuffles per container + cumulative reshuffles (logistics-style)
# ----------------------------

# Count how many times each container appears as a blocked "lower" container
block_counts = {c["id"]: 0 for c in containers}
if len(blocking_df) > 0:
    for _, r in blocking_df.iterrows():
        block_counts[r["lower"]] += 1

# Retrieval order = increasing departure time
retrieve_order = sorted([c["id"] for c in containers], key=lambda cid: cid_to_d[cid])
resh_per = [block_counts[cid] for cid in retrieve_order]

# Bar chart
plt.figure(figsize=(7, 3.2))
plt.bar(retrieve_order, resh_per, color="#2E90FA", edgecolor="#101828", alpha=0.85)
plt.title("Tier 3 GA — reshuffles needed per container (retrieval order)")
plt.xlabel("Container")
plt.ylabel("Reshuffles")
plt.grid(True, axis="y", alpha=0.25)
plt.show()

# Cumulative
cum = np.cumsum(resh_per)
plt.figure(figsize=(7, 3.2))
plt.plot(retrieve_order, cum, marker="o", lw=2, color="#344054")
plt.title("Tier 3 GA — cumulative reshuffles")
plt.xlabel("Container (retrieval order)")
plt.ylabel("Cumulative reshuffles")
plt.grid(True, alpha=0.25)
plt.show()

pd.DataFrame({"container": retrieve_order, "d": [cid_to_d[c] for c in retrieve_order], "reshuffles": resh_per})

## Solution quality check: how close is GA to the exact optimum (on this small instance)?

Because this example is small, we can compute the **true optimum reshuffle count** by exhaustive enumeration (Tier 1 style).

Then we compare:

- the best reshuffles found by GA
- the optimal reshuffles

This makes the quality gap (if any) explicit.

In [ ]:
# Compute the exact optimum reshuffles (Tier 1 style) and compare with GA.
#
# This is only feasible because the instance is small:
# 5 containers, 3 stacks, 2 tiers.

import itertools


def apply_choice_sequence(choice_seq: Tuple[int, ...]) -> Optional[List[List[str]]]:
    stacks: List[List[str]] = [[] for _ in range(NUM_STACKS)]
    for cid, s_idx in zip(arrival_order, choice_seq):
        if len(stacks[s_idx]) >= MAX_TIERS:
            return None
        stacks[s_idx].append(cid)
    return stacks


all_choice_sequences = list(itertools.product(range(NUM_STACKS), repeat=len(arrival_order)))

opt_resh = None
opt_cfg = None

for seq in all_choice_sequences:
    cfg = apply_choice_sequence(seq)
    if cfg is None:
        continue
    resh = int(count_total_reshuffles(cfg))
    if opt_resh is None or resh < opt_resh:
        opt_resh = resh
        opt_cfg = cfg

assert opt_resh is not None

# GA result reshuffles
best_resh_ga = int(count_total_reshuffles(best_config))

print("Optimal reshuffles (exact):", int(opt_resh))
print("GA reshuffles:", int(best_resh_ga))
print("Gap (GA - optimal):", int(best_resh_ga) - int(opt_resh))
print("One optimal config:", opt_cfg)

Optimal reshuffles (exact): 0
GA reshuffles: 0
Gap (GA - optimal): 0
One optimal config: [['C1', 'C2'], ['C3', 'C4'], ['C5']]


## Why this Tier exists vs earlier Tiers (comparison)

### Why Tier 3 exists (vs Tier 1–2)

- Tier 1 shows what “optimal” means, but exact optimization becomes infeasible as the yard gets larger.
- Tier 2 is fast and interpretable, but can be short-sighted.

Tier 3 exists to search a much larger space of stacking plans than greedy heuristics, without requiring an exact solver.

### Advantages vs Tier 1

- Scales better: you can cap runtime by choosing population size and generations.
- Flexible: you can modify the fitness function to reflect real operational rules.

### Advantages vs Tier 2

- Less greedy: can escape local decisions via mutation/crossover.
- Often finds better solutions if you allow some compute time.

### Disadvantages

- Not guaranteed optimal.
- Requires parameter tuning (population size, mutation rate, generations).
- Less “rule-like” than Tier 2 (harder to explain a GA to operators).

### When to use Tier 3

- You can afford seconds/minutes of compute time.
- You want better solution quality than a simple heuristic.
- You want a flexible optimizer that can incorporate additional penalties/constraints.